In [1]:

import sys
import logging
sys.path.append('E:\wangzhilin\QuantumGalaxy')
from QGI.mysql import MYSQL

import pandas as pd
import numpy as np
from QGI.neoapi import Neo4j
from QGI.feishu import *

In [5]:
from QGI import neo2mysql
neo2mysql.main()

08-24 16:41 logger       INFO     using username Local_Editor
08-24 16:41 logger       INFO     accessed to database <qgdbs>
08-24 16:41 logger       INFO     accept cypher query match (n) return id(n),labels(n),n.name,n.code
08-24 16:41 logger       INFO     get 5346 records
08-24 16:41 logger       INFO     got a cursor
08-24 16:41 logger       INFO     executed and commited 0 sql "insert into nodes (labels,name,code,id,using_flag) values(%s,%s,%s,%s,1) on duplicate key update labels=%s,name=%s,code=%s,using_flag=1"
08-24 16:41 logger       INFO     got a cursor
08-24 16:41 logger       INFO     executed and commited 0 sql "insert into nodes_records (id,field,record_index,conclusion) values(%s,%s,0,%s) on duplicate key update conclusion=%s"
08-24 16:41 logger       INFO     got a cursor
08-24 16:41 logger       INFO     executed and commited 0 sql "insert into nodes_records (id,field,record_index,conclusion) values(%s,%s,0,%s) on duplicate key update conclusion=%s"
08-24 16:41 logger

In [3]:
def get_logger(name,level=None,file=None):
    logger=logging.getLogger(name)
    logger.setLevel(logging.INFO)
    
    ch=logging.StreamHandler()
    ch.setLevel(logging.INFO)
    formatter=logging.Formatter(fmt='%(asctime)s %(name)-12s %(levelname)-8s %(message)s',datefmt='%m-%d %H:%M')
    ch.setFormatter(formatter)
    
    logger.addHandler(ch)
    if file:    
        fh=logging.FileHandler("E:/wangzhilin/QuantumGalaxy/logs/%s.log"%file,'a',encoding='utf-8')
        fh.setLevel(logging.INFO)
        fh.setFormatter(formatter)
        logger.addHandler(fh)
    return logger


In [4]:
logger=get_logger('logger',file=None)

In [6]:
logger.info('?')


08-24 16:42 logger       INFO     ?


In [130]:
host='localhost'
user='Local_Editor'
password='QuantumGalaxy'
database='qgdbs'
mysql=MYSQL(host,user,password,database)

08-29 13:47 logger       INFO     using username Local_Editor
08-29 13:47 logger       INFO     accessed to database <qgdbs>


In [8]:
mysql.close()

08-24 16:42 logger       INFO     Closed connection to database <qgdbs>


In [9]:
class FeishuException(Exception):

    pass

In [10]:
class F(FeishuAPI):
    def _get_table_metainfo(self,table_token):
        headers = {
            'Authorization':
            'Bearer ' + self._fetch_tenant_token(),  # your access token
            'Content-Type': 'application/json'
        }
        url='https://open.feishu.cn/open-apis/sheets/v2/spreadsheets/'+table_token+'/metainfo'
        params = {"user_id_type": "open_id"}
        response = requests.request("GET",
                                    url,
                                    params=params,
                                    headers=headers,
                                    )
        if response.status_code==200:
        
            return response
        else:
            msg='Failed to get a response, response code :%d'%response.status_code
            '''404 大概率token错，403表格没加权限'''
            logger.error(msg)
            #return response
            raise FeishuException(msg)

    def _get_sheets_id_rowcount(self,table_token):
        metadata=self._get_table_metainfo(table_token)
        if metadata.json()['code']==0:
            sheets=[]
            for sheet in metadata.json()['data']['sheets']:
                sheets.append((sheet['title'],sheet['sheetId'],sheet['rowCount']))
                #sheets[sheet['title']]=sheet['sheetId']
            return sheets

        else:
            msg='Failed to get metainfo, server error code: %d'%metadata.json()['code']
            logger.error(msg)
            raise FeishuException(msg)
    
    def get_records(self,table_token):
        sheets=self._get_sheets_id_rowcount(table_token)
        #sheets:[(name,id,rowcount)]

        node_sheet_id=sheets[0][1]
        edge_sheet_id=sheets[1][1]
        url1='https://open.feishu.cn/open-apis/sheets/v2/spreadsheets/'+table_token+'/values/'+node_sheet_id+'!A2:L'
        url2='https://open.feishu.cn/open-apis/sheets/v2/spreadsheets/'+table_token+'/values/'+edge_sheet_id+'!A2:N'
        #A2：L 返回到最大非空行为止
        headers = {
            'Authorization':
            'Bearer ' + self._fetch_tenant_token(),  # your access token
            'Content-Type': 'application/json'
        }
        params={'dateTimeRenderOption':'FormattedString' }
        '''nodes'''
        r1 = requests.request("GET",
                                    url1,
                                    params=params,
                                    headers=headers,
                                    )
        if r1.status_code==200:
            if r1.json()['code']==0:
                nodes=r1.json()['data']['valueRange']['values']
            else:
                msg='Failed to get metainfo, server error code: %d'%r1.json()['code']
                logger.error(msg)
                raise FeishuException(msg)
        else:
            msg='Failed to get a response, response code :%d'%r1.status_code
            
            logger.error(msg)
            #return response
            raise FeishuException(msg)
        '''edges'''
        r2 = requests.request("GET",
                                    url2,
                                    params=params,
                                    headers=headers,
                                    )
        if r2.status_code==200:
            if r2.json()['code']==0:
                edges=r2.json()['data']['valueRange']['values']
            else:
                msg='Failed to get metainfo, server error code: %d'%r2.json()['code']
                logger.error(msg)
                raise FeishuException(msg)
        else:
            msg='Failed to get a response, response code :%d'%r2.status_code
            
            logger.error(msg)
            #return response
            raise FeishuException(msg)
        return nodes,edges

        


In [11]:
ch=['非标的上下游','别称','概念图','一句话描述','所属行业','人工费用','能源费用','财务利息费用','营销研发费用','生产产能','生产产量','生产产值','生产地区','销售地区','销量','产品营收','销售占比','扩产难度','产品售价','销售模式','生产模式','产品规格','产品单位','成本占比','配方系数','去向占比','供应内容','竞争优势','竞争劣势','市场规模','竞争格局','竞争对手','原材料','下游应用','最新拆解市值','认为销量','认为单位利润','世界线','财报数据','行情数据','毛利率（%）']
eg=['other_supplier_info','nick_name','main_picture','summary','industry','labor_cost','energy_cost','financial_cost','rd_cost','produce_capacity','produce_volume','produce_value','produce_region','sale_region','sale_volume','sale_value','sale_ratio','expand_difficulty','sell_price','sell_pattern','produce_pattern','product_spec','product_unit','in_ratio','in_recipe','out_ratio','supply_content','advantage','disadvantage','market_scale','competitive_condition','competitor','raw_material','downstream','comprehend_mv','comprehend_unit_profit','comprehend_volume','world_line','financial_data','market_data','gross_profit']
field_dict=dict(zip(ch,eg))


In [12]:
print(field_dict)

{'非标的上下游': 'other_supplier_info', '别称': 'nick_name', '概念图': 'main_picture', '一句话描述': 'summary', '所属行业': 'industry', '人工费用': 'labor_cost', '能源费用': 'energy_cost', '财务利息费用': 'financial_cost', '营销研发费用': 'rd_cost', '生产产能': 'produce_capacity', '生产产量': 'produce_volume', '生产产值': 'produce_value', '生产地区': 'produce_region', '销售地区': 'sale_region', '销量': 'sale_volume', '产品营收': 'sale_value', '销售占比': 'sale_ratio', '扩产难度': 'expand_difficulty', '产品售价': 'sell_price', '销售模式': 'sell_pattern', '生产模式': 'produce_pattern', '产品规格': 'product_spec', '产品单位': 'product_unit', '成本占比': 'in_ratio', '配方系数': 'in_recipe', '去向占比': 'out_ratio', '供应内容': 'supply_content', '竞争优势': 'advantage', '竞争劣势': 'disadvantage', '市场规模': 'market_scale', '竞争格局': 'competitive_condition', '竞争对手': 'competitor', '原材料': 'raw_material', '下游应用': 'downstream', '最新拆解市值': 'comprehend_mv', '认为销量': 'comprehend_unit_profit', '认为单位利润': 'comprehend_volume', '世界线': 'world_line', '财报数据': 'financial_data', '行情数据': 'market_data', '毛利率（%）': 'gross_profit'}


In [140]:

import copy
class Loader():
    def __init__(self,nodes,edges):
        self.nodes=copy.deepcopy(nodes)
        self.edges=copy.deepcopy(edges)
        self.error=[]
        pass
    def _get_mysql(self):
        
        host='localhost'
        user='Local_Editor'
        password='QuantumGalaxy'
        database='qgdbs'
        self.mysql=MYSQL(host,user,password,database)
        return self.mysql
    def _close_mysql(self):
        return self.mysql.close()
    def _get_neo(self):
        uri2='neo4j+ssc://bd2c184f.databases.neo4j.io'
        user2='neo4j'
        password2='OkvWgegnLgUSuAPl-Zcu8wkcXCX-GxNRC2zydwpbwrE'
        #self.neo=Neo4j(uri2,user2,password2)
        uri = "neo4j+ssc://08ef0a79.databases.neo4j.io"
        user = "QG_Editor"
        password = "editor"
        self.neo=Neo4j(uri,user,password)
        return self.neo
    def _close_neo(self):
        return self.neo.close()
    def get_node_record_index(self,node_id,field,main=False):
        if main:
            return 0
        sql='select count(*) from nodes_records where id=%s and field like "%%%s%%" and not record_index=0'%(node_id,field)
        #print(sql)
        r=self.mysql.read_query(sql)[0][0]+1
        
        return r
    def load_node_record(self,record,columns=None,main=False):
        #if none, record [name,label,field,conclusion,value,start_date,end_date,source,demonstration,region,currency,writer]
        #convert name+label to id, generate record_index
        
        name=record.pop(0)
        label=record.pop(0)
        '''if_main=record.pop(1)
        if if_main=='是':
            if_main=True
        else:
            if_main=False
        print(if_main)'''
        #print(name)
        #print(label)
        r=self.mysql.read_query('select id from nodes where name ="%s" and labels like "%%%s%%"'%(name,label))
        if len(r)==0:
            msg='found no node which name is "%s" and labels contains "%s"'%(name,label)
            logger.error(msg)
            self.error.append(msg)
            raise Exception(msg)
        elif len(r)>1:
            msg='found mutiple nodes which name is "%s" and labels contains "%s"'%(name,label)
            logger.error(msg)
            self.error.append(msg)
            raise Exception(msg)
        else:
            id=r[0][0]
            record.insert(0,id)

            
            record[1]=field_dict[record[1]]
            record_index=self.get_node_record_index(record[0],record[1],main=main)
            record.insert(1,record_index)
            return record
    def load_node_records(self,node_records,columns=None,main=False):

        '''由于获取recordindex不能一次性，必须每条record单独录入'''
        self.nodes_id=[]
        for r in node_records:
            try:
                val=[]
                val.append(tuple(self.load_node_record(r,columns,main=main)))
                '''todo :事务性机制，避免多次插入'''
                sql='insert ignore into nodes_records (id,record_index,field,conclusion,value,start_date,end_date,source,demonstration,region,currency,writer) values (%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s) '
                self.mysql.write_many_query_without_commit(sql,val)
                if val[0][0] not in self.nodes_id:
                    self.nodes_id.append(val[0][0])
            except:
                pass
        
        return True
    def get_edge_record_index(self,start,end,field,main=False):
        if main:
            return 0
        sql='select count(*) from edges_records where start=%s and end =%s and field like "%%%s%%" and not record_index=0'%(start,end,field)
        #print(sql)
        r=self.mysql.read_query(sql)[0][0]+1
        
        return r
    def load_edge_record(self,record,columns=None,main=False):
        #if none, record [name,label,field,conclusion,value,start_date,end_date,source,demonstration,region,currency,writer]
        #convert name+label to id, generate record_index
        
        start_name=record.pop(0)
        start_label=record.pop(0)
        end_name=record.pop(0)
        end_label=record.pop(0)

        #print(name)
        #print(label)
        s=self.mysql.read_query('select id from nodes where name ="%s" and labels like "%%%s%%"'%(start_name,start_label))
        e=self.mysql.read_query('select id from nodes where name ="%s" and labels like "%%%s%%"'%(end_name,end_label))
        if len(s)==0:
            msg='found no node which name is "%s" and labels contains "%s"'%(start_name,start_label)
            logger.error(msg)
            self.error.append(msg)
            raise Exception(msg)
        elif len(s)>1:
            msg='found mutiple nodes which name is "%s" and labels contains "%s"'%(start_name,start_label)
            logger.error(msg)
            self.error.append(msg)
            raise Exception(msg)
        
        else:
            s=s[0][0]
            if len(e)==0:
                msg='found no node which name is "%s" and labels contains "%s"'%(end_name,end_label)
                logger.error(msg)
                self.error.append(msg)
                raise Exception(msg)
            elif len(e)>1:
                msg='found mutiple nodes which name is "%s" and labels contains "%s"'%(end_name,end_label)
                logger.error(msg)
                self.error.append(msg)
                raise Exception(msg)
            else:
                e=e[0][0]
                r=self.mysql.read_query('select id from edges where start="%s" and end ="%s"'%(s,e))
                if len(r)==0:
                    msg='found no edge which start is "%s" and end is "%s"'%(start_name,end_name)
                    logger.error(msg)
                    self.error.append(msg)
                    raise Exception(msg)
                elif len(r)>1: 
                    msg='found mutiple edges which name is "%s" and end is "%s"'%(start_name,end_name)
                    logger.error(msg)
                    self.error.append(msg)
                    raise Exception(msg)
                else:
                    id=r[0][0]
                    record.insert(0,e)
                    record.insert(0,s)
                    #print(record)
                    
                    record[2]=field_dict[record[2]]
                    record_index=self.get_edge_record_index(s,e,record[2],main=main)
                    
                    record.insert(2,record_index)
                    return record
    def load_edge_records(self,edge_records,columns=None,main=False):
        '''由于获取recordindex不能一次性，必须每条record单独录入'''
        self.edges_id=[]
        for r in edge_records:
            try:
                val=[]
                val.append(tuple(self.load_edge_record(r,columns,main=main)))
                sql='insert ignore into edges_records (start,end,record_index,field,conclusion,value,start_date,end_date,source,demonstration,region,currency,writer) values (%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s) '
                #print(sql)
                self.mysql.write_many_query_without_commit(sql,val)
                if (val[0][0],val[0][1]) not in self.edges_id:
                    self.edges_id.append((val[0][0],val[0][1]))
                
            except:
                pass
        
        return True
    def update_neo(self):
        #只更新本次录入的部分
        self.nodes_id=list(set(self.nodes_id))
        self.edges_id=list(set(self.edges_id))
        for node in self.nodes_id:
            sql='select field,conclusion from nodes_records where id=%s and record_index=1'%node
            records=self.mysql.read_query(sql)
            cypher='match (n) where id(n)=%s set'%node
            if len(records)>0:
                for r in records:
                    cypher+=' n.%s="%s",'%(r[0],r[1])
                    #print(cypher)
                cypher=cypher[:-1]
                self.neo.simple_query(cypher)
        for edge in self.edges_id:
            sql='select field,conclusion from edges_records where start=%s and end =%s and record_index=1'%(edge[0],edge[1])
            records=self.mysql.read_query(sql)
            cypher='match (n)-[r]-(m) where id(n)=%s and id(m)=%s set'%(edge[0],edge[1])
            if len(records)>0:
                for r in records:
                    cypher+=' r.%s="%s",'%(r[0],r[1])
                cypher=cypher[:-1]
                r=self.neo.simple_query(cypher)
                
            
        pass
    def update_neo_full(self):
        sql='select id,field,conclusion from nodes_records where  record_index=1 and not field in ("name","code")'
        records=self.mysql.read_query(sql)
        for r in records:
            id,field,conclusion=r
            cypher='match (n) where id(n)=%s set n.%s="%s"'%(id,field,conclusion)
            self.neo.simple_query(cypher)
        sql='select start,end,field,conclusion from edges_records where record_index=1'
        records=self.mysql.read_query(sql)
        for r in records:
            start,end,field,conclusion=r
            cypher='match (n)-[r]-(m) where id(n)=%s and id(m)=%s set r.%s="%s"'%(start,end,field,conclusion)
            self.neo.simple_query(cypher)
    def load(self,main=False):
        neo2mysql.main()
        self._get_mysql()
        self._get_neo()
        self.load_node_records(self.nodes,main=main)
        self.load_edge_records(self.edges,main=main)
        if len(self.error)==0:
            self.mysql._commit()
            self.update_neo()
            self.msg='finished'
        else:
            self.msg='has error and rollback:\n'
            for e in self.error:
                self.msg+=str(e)+'\n'
        self._close_mysql()
        self._close_neo()
        #print(self.nodes_id)
        #print(self.edges_id)
        return self.msg
        



In [141]:
token='shtcnm1CVnaTGYeWIzbLf8bmIch'

feishu=F()
node,edge=feishu.get_records(token)
print(len(node),len(edge))


0 0


In [142]:
node[0]

IndexError: list index out of range

In [143]:

loader=Loader(node,edge)
loader._get_mysql()
loader._get_neo()


08-29 13:52 logger       INFO     using username Local_Editor
08-29 13:52 logger       INFO     accessed to database <qgdbs>


In [144]:
loader.update_neo_full()

08-29 13:52 logger       INFO     got a cursor
08-29 13:52 logger       INFO     Accepted read sql query "select id,field,conclusion from nodes_records where  record_index=1 and not field in ("name","code")"
08-29 13:52 logger       INFO     accept cypher query match (n) where id(n)=9 set n.advantage="主营煤炭毛利高于同业可比公司，21年中报煤炭毛利对比：华阳股份20.84%，神火股份31.03%，兰花科创58.2%。 "
08-29 13:52 logger       INFO     get 0 records
08-29 13:52 logger       INFO     accept cypher query match (n) where id(n)=9 set n.disadvantage="产能差距：兰花科创无烟煤核定产能900万吨每年， ；华阳股份,核定在产产能3270 万吨,权益产能2836万吨(占比 87%),其中大部分为稀缺煤种无烟煤。
2021年中国尿素产能第一为山西天泽（290万吨，占比4.03%），第四为华鲁恒升（155万吨，占比2.15%），兰花科创为年尿素产能 100 万吨。"
08-29 13:52 logger       INFO     get 0 records
08-29 13:52 logger       INFO     accept cypher query match (n) where id(n)=9 set n.financial_cost="财务利息费用为5.23亿元"
08-29 13:52 logger       INFO     get 0 records
08-29 13:52 logger       INFO     accept cypher query match (n) where id(n)=9 set n.industry="煤炭、化肥化工"
08-29 13:52 logger 

In [126]:

msg=loader.load()
print(msg)



08-29 13:39 logger       INFO     using username Local_Editor
08-29 13:39 logger       INFO     accessed to database <qgdbs>
08-29 13:39 logger       INFO     accept cypher query match (n) return id(n),labels(n),n.name,n.code
08-29 13:39 logger       INFO     get 5550 records
08-29 13:39 logger       INFO     got a cursor
08-29 13:39 logger       INFO     executed and commited 0 sql "insert into nodes (labels,name,code,id,using_flag) values(%s,%s,%s,%s,1) on duplicate key update labels=%s,name=%s,code=%s,using_flag=1"
08-29 13:39 logger       INFO     got a cursor
08-29 13:39 logger       INFO     executed and commited 0 sql "insert into nodes_records (id,field,record_index,conclusion) values(%s,%s,0,%s) on duplicate key update conclusion=%s"
08-29 13:39 logger       INFO     got a cursor
08-29 13:39 logger       INFO     executed and commited 0 sql "insert into nodes_records (id,field,record_index,conclusion) values(%s,%s,0,%s) on duplicate key update conclusion=%s"
08-29 13:39 logger

finished


In [132]:
r=mysql.read_query('select id,field,conclusion from nodes_records limit 5')
r

08-29 13:47 logger       INFO     got a cursor
08-29 13:47 logger       INFO     Accepted read sql query "select id,field,conclusion from nodes_records limit 5"


((0, 'code', None),
 (0, 'name', '电力需求'),
 (1, 'code', '300140.SZ'),
 (1, 'name', '中环装备'),
 (2, 'code', 'OAS.O'))

In [133]:
for record in r:
    a,b,c=record
    print(b)

code
name
code
name
code


In [23]:
neo=loader._get_neo()


In [24]:
neo.read_query('match (n {name:"鹿山新材"}) return n.code')

[<Record n.code='603051.SH'>]

In [14]:
main()

08-11 14:17 logger       ERROR    Failed to get a response, response code :403


FeishuException: Failed to get a response, response code :403